# Statistical Tests: Do better-looking listing photos help Airbnb hosts charge more?

This notebook compares two models from `OUTPUT/models/model_specs.md`:
- **M1 (baseline):** listing details only
- **M2 (photo model):** M1 + three standardized photo quality scores (MUSIQ-KonIQ, MUSIQ-AVA, CLIP-IQA)

Steps:
1. VIF check for correlated photo scores (MI2 feedback)
2. Partial F-test: do the photo scores improve M1? (α = 0.05)
3. Two-sided t-tests for each photo score (α = 0.05)
4. ΔR² check on the test set (goal: ΔR² ≥ 0.05)
5. 95% bootstrap confidence interval for ΔR² (1,000 resamples)

**Input:** `DATA/final/analysis_data.csv`. In Colab, upload it with the folder icon on the left, then Runtime → Run all.

In [ ]:
DATA_FILE  = "analysis_data.csv"
ALPHA      = 0.05
DELTA_GOAL = 0.05
N_BOOT     = 1000
SEED       = 42
VIF_CUTOFF = 5

import numpy as np, pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

## Load data and fit M1 and M2 (training rows, HC3 robust standard errors, as in model_specs.md)

In [ ]:
df = pd.read_csv(DATA_FILE, low_memory=False)
train = df[df["split"] == "train"].copy()
test  = df[df["split"] == "test"].copy()
print(f"Train: {len(train)}  Test: {len(test)}")

BASE  = ("log_price ~ private_room + bedrooms + bathrooms + shared_bath + accommodates"
         " + brooklyn + queens + bronx_si + min30 + private_room:bedrooms")
PHOTO = ["z_musiq_koniq", "z_musiq_ava", "z_clipiqa"]

m1 = smf.ols(BASE, data=train).fit(cov_type="HC3")
m2 = smf.ols(BASE + " + " + " + ".join(PHOTO), data=train).fit(cov_type="HC3")
print(f"Training R²: M1 = {m1.rsquared:.4f}, M2 = {m2.rsquared:.4f}")

## 1. VIF check (MI2 feedback)
All three scores measure photo quality, so they may be correlated. A VIF above 5 would make the individual t-tests unreliable.

In [ ]:
print("Correlation of photo scores (training set):")
display(train[PHOTO].corr().round(3))

X = train[["private_room", "bedrooms", "bathrooms", "shared_bath", "accommodates",
           "brooklyn", "queens", "bronx_si", "min30"] + PHOTO].astype(float)
X["private_room:bedrooms"] = X["private_room"] * X["bedrooms"]
X = sm.add_constant(X)
vif = pd.DataFrame({"variable": X.columns,
                    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
vif = vif[vif["variable"] != "const"].reset_index(drop=True)
display(vif.round(2))
max_vif = vif.loc[vif["variable"].isin(PHOTO), "VIF"].max()
print(f"Highest photo-score VIF = {max_vif:.2f} ->", "PROBLEM (> 5)" if max_vif > VIF_CUTOFF else "OK (< 5)")

## 2. Partial F-test (M2 vs M1)
H0: all three photo score coefficients = 0. One-sided by design (only asks whether fit improves).
Uses a robust Wald F-test, since the models use HC3 robust standard errors.

In [ ]:
w = m2.wald_test(" = 0, ".join(PHOTO) + " = 0", use_f=True, scalar=True)
F, p_F, df1, df2 = float(w.statistic), float(w.pvalue), int(w.df_num), int(w.df_denom)
f_test = pd.DataFrame([{"Test": "Partial F-test (M2 vs M1)", "F": F, "df": f"({df1}, {df2})",
                        "p": p_F, "Significant (p < 0.05)": p_F < ALPHA}])
display(f_test.round(4))

## 3. Two-sided t-tests for each photo score
Coefficient = change in log price for a 1-SD higher score, holding listing details constant.
`pct_change` = 100 × (e^β − 1), the approximate % change in nightly price.

In [ ]:
ci = m2.conf_int(alpha=ALPHA)
t_tests = pd.DataFrame({
    "score": PHOTO,
    "coef": m2.params[PHOTO].values,
    "robust_se": m2.bse[PHOTO].values,
    "t": m2.tvalues[PHOTO].values,
    "p (two-sided)": m2.pvalues[PHOTO].values,
    "ci_low": ci.loc[PHOTO, 0].values,
    "ci_high": ci.loc[PHOTO, 1].values,
})
t_tests["pct_change"] = 100 * (np.exp(t_tests["coef"]) - 1)
t_tests["significant"] = t_tests["p (two-sided)"] < ALPHA
display(t_tests.round(4))

## 4. ΔR² check on the test set (goal: ΔR² ≥ 0.05)

In [ ]:
def r2(y, yhat):
    return 1 - np.sum((y - yhat) ** 2) / np.sum((y - y.mean()) ** 2)

y = test["log_price"].values
p1, p2 = m1.predict(test).values, m2.predict(test).values
r2_m1, r2_m2 = r2(y, p1), r2(y, p2)
delta = r2_m2 - r2_m1
print(f"Test R²: M1 = {r2_m1:.4f}, M2 = {r2_m2:.4f}")
print(f"ΔR² = {delta:.4f} ->", "meets goal (≥ 0.05)" if delta >= DELTA_GOAL else "below 0.05 goal")

## 5. 95% bootstrap confidence interval for ΔR²
Resample the test set with replacement 1,000 times, recompute ΔR² each time, and take the 2.5th and 97.5th percentiles.

In [ ]:
rng = np.random.default_rng(SEED)
n = len(y)
boot = np.empty(N_BOOT)
for b in range(N_BOOT):
    i = rng.integers(0, n, n)
    boot[b] = r2(y[i], p2[i]) - r2(y[i], p1[i])
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
print(f"ΔR² = {delta:.4f}, 95% bootstrap CI [{ci_low:.4f}, {ci_high:.4f}] ->",
      "excludes 0" if ci_low > 0 else "includes 0")

## Summary and saved outputs

In [ ]:
summary = pd.DataFrame({
    "Criterion": ["VIF of photo scores < 5", "Partial F-test p < 0.05", "Test ΔR² ≥ 0.05", "95% bootstrap CI excludes 0"],
    "Result": [f"max VIF = {max_vif:.2f}", f"F({df1}, {df2}) = {F:.2f}, p = {p_F:.4g}",
               f"ΔR² = {delta:.4f} (M1 {r2_m1:.4f} → M2 {r2_m2:.4f})", f"[{ci_low:.4f}, {ci_high:.4f}]"],
    "Met?": [max_vif < VIF_CUTOFF, p_F < ALPHA, delta >= DELTA_GOAL, ci_low > 0]})
display(summary)

vif.round(3).to_csv("vif.csv", index=False)
f_test.round(4).to_csv("f_test.csv", index=False)
t_tests.round(4).to_csv("t_tests.csv", index=False)
summary.to_csv("hypothesis_test_summary.csv", index=False)
pd.DataFrame({"delta_r2": boot}).to_csv("bootstrap_samples.csv", index=False)
print("Saved: vif.csv, f_test.csv, t_tests.csv, hypothesis_test_summary.csv, bootstrap_samples.csv")